In [58]:
# !pip install openai
#!pip install groq

In [7]:
import os

import dotenv

dotenv.load_dotenv(dotenv_path='C:\\Users\\rcwoo\\PycharmProjects\\data5580_hw\\.env')

True

In [8]:
# Importing necessary libraries
import numpy as np
import uuid
import umap
import shap
import pandas as pd
import faker
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import classification_report, accuracy_score
import datetime

from arize.pandas.logger import Client
from arize.utils.types import ModelTypes, Environments, Schema, Metrics, Embedding, EmbeddingColumnNames
import numpy as np
import pandas as pd
from faker import Faker


RANDOM_STATE = 123
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

def get_uuid():
    return uuid.uuid4().hex 
    
def random_dates(start, end, n=10):

    start_u = start.value//10**9
    end_u = end.value//10**9
    
    return pd.to_datetime(np.random.randint(start_u, end_u, n), unit='s')

fake = Faker()

In [9]:
# Load the breast cancer dataset
data = load_breast_cancer()

# Converting the dataset into a pandas DataFrame for easier handling
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target  # Add the target column (0 for malignant, 1 for benign)
df['target'] = df['target'].map({0: 'benign', 1: 'malignant'}) 

In [10]:
# Create a primary key for the dataset
df['prediction_id'] = [get_uuid() for _ in range(df.shape[0])]

start = pd.to_datetime('2025-03-01')
end = pd.to_datetime('2025-03-15')

df['event_timestamp'] = random_dates(start, end, df.shape[0])

df['state'] = [fake.state() for _ in range(df.shape[0])]

In [11]:
# Splitting the dataset into training and testing sets (80% training, 20% testing)
X = df.drop('target', axis=1)  # Features
y = df['target']               # Target

features = X.drop(['prediction_id', 'event_timestamp', 'state'], axis=1).columns

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

In [12]:
# Initialize the Random Forest Classifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)

# Fit the model to the training data
rf_classifier.fit(X_train[features], y_train)

# Make predictions on the test set
y_pred = rf_classifier.predict(X_test[features])

# Evaluate the model's performance using accuracy and classification report
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9912

Classification Report:
              precision    recall  f1-score   support

      benign       1.00      0.98      0.99        41
   malignant       0.99      1.00      0.99        73

    accuracy                           0.99       114
   macro avg       0.99      0.99      0.99       114
weighted avg       0.99      0.99      0.99       114



In [13]:
# Create the SHAP explainer for the trained model
explainer = shap.TreeExplainer(rf_classifier)

# Calculate SHAP values for the test set
shap_values = explainer.shap_values(X_test[features])

shap_values_positive_class = shap_values[:,:,1]

# 6. Convert the SHAP values to a DataFrame
shap_df = pd.DataFrame(shap_values_positive_class, columns=features)

# Because we are doing a bulk upload we need to rename the columns by added _shap at the end
shap_dataframe = pd.DataFrame(
        shap_values_positive_class, columns=[f"{fn}_shap" for fn in features], index=X_test.index
)

# Create a mapping
mapping = {fn: f"{fn}_shap" for fn in X[features].columns}

shap_dataframe.head()

,mean radius_shap,mean texture_shap,mean perimeter_shap,mean area_shap,mean smoothness_shap,mean compactness_shap,mean concavity_shap,mean concave points_shap,mean symmetry_shap,mean fractal dimension_shap,...,worst radius_shap,worst texture_shap,worst perimeter_shap,worst area_shap,worst smoothness_shap,worst compactness_shap,worst concavity_shap,worst concave points_shap,worst symmetry_shap,worst fractal dimension_shap
333,0.006227,0.012070,0.010931,0.011793,0.002703,0.002851,0.023857,0.051082,0.000684,0.000399,...,0.046181,0.008758,0.039769,0.054636,0.005563,0.007451,0.019355,0.041595,0.001851,0.000665
273,0.008022,0.007974,0.013161,0.017900,0.001117,0.002766,0.023050,0.047972,-0.000969,0.000877,...,0.047510,0.013139,0.043722,0.056637,-0.002086,0.006957,0.020438,0.042495,0.001245,0.001043
201,-0.023076,-0.002633,-0.033997,-0.033522,0.001392,-0.001851,-0.030186,-0.087250,-0.000824,-0.002244,...,-0.095546,-0.000588,-0.065963,-0.108055,0.000095,-0.003081,-0.020623,-0.088405,-0.000041,-0.000516
178,0.009886,-0.020028,0.009680,0.011953,0.009469,0.003560,0.024270,0.060509,0.001172,-0.007354,...,0.044889,0.000599,0.034879,0.049116,0.014614,0.007606,0.018536,0.041735,0.006710,-0.001883
85,-0.023417,-0.000158,-0.031711,-0.032024,-0.000701,0.002015,-0.035971,-0.085456,-0.001582,-0.000027,...,-0.094815,-0.006637,-0.067245,-0.110220,-0.001136,-0.001609,-0.015189,-0.073418,-0.002053,0.000660


In [14]:
embedder = umap.UMAP(random_state=RANDOM_STATE, n_components=3)

X_train_embeddings = embedder.fit_transform(X_train[features], y_train.map({'benign': 0, 'malignant': 1}))

In [15]:
from groq import Groq

In [16]:
client = Groq(api_key=os.environ["GROQ_API_KEY"])
completion = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
      {
        "role": "user",
        "content": "Hello, what model are you?"
      }
    ],
    temperature=1,
    max_completion_tokens=800,
    top_p=1,
    # reasoning_effort="low",
    stream=True,
    stop=None
)

for chunk in completion:
    print(chunk.choices[0].delta.content or "", end="")


I'm an AI model known as Llama. Llama stands for "Large Language Model Meta AI."

In [2]:
# from openai import OpenAI
#
# client = OpenAI(
#   api_key=API_KEY
# )
#
# completion = client.chat.completions.create(
#   model="gpt-4o-mini",
#   store=True,
#   messages=[
#     {"role": "user", "content": "write a haiku about ai"}
#   ]
# )
#
# print(completion.choices[0].message);

ChatCompletionMessage(content='Silent circuits hum,  \nThoughts woven from code and light,  \nDreams in binary.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


In [17]:
X_train.iloc[0].to_dict()

{'mean radius': 14.22,
 'mean texture': 23.12,
 'mean perimeter': 94.37,
 'mean area': 609.9,
 'mean smoothness': 0.1075,
 'mean compactness': 0.2413,
 'mean concavity': 0.1981,
 'mean concave points': 0.06618,
 'mean symmetry': 0.2384,
 'mean fractal dimension': 0.07542,
 'radius error': 0.286,
 'texture error': 2.11,
 'perimeter error': 2.112,
 'area error': 31.72,
 'smoothness error': 0.00797,
 'compactness error': 0.1354,
 'concavity error': 0.1166,
 'concave points error': 0.01666,
 'symmetry error': 0.05113,
 'fractal dimension error': 0.01172,
 'worst radius': 15.74,
 'worst texture': 37.18,
 'worst perimeter': 106.4,
 'worst area': 762.4,
 'worst smoothness': 0.1533,
 'worst compactness': 0.9327,
 'worst concavity': 0.8488,
 'worst concave points': 0.1772,
 'worst symmetry': 0.5166,
 'worst fractal dimension': 0.1446,
 'prediction_id': 'e15a64ef810e47a698bbd45ec4bbe2d7',
 'event_timestamp': Timestamp('2025-03-11 20:35:11'),
 'state': 'New York'}

In [31]:
y_train[0]

'benign'

In [18]:
from scipy.spatial.distance import cdist

def closest_node(node, nodes):
    return nodes[cdist([node], nodes).argmin()]

In [19]:
target = closest_node(X_train_embeddings[0], X_train_embeddings[1:])
target

array([32.25736  , -1.0761801, 14.214847 ], dtype=float32)

In [20]:
np.where(X_train_embeddings == target)

(array([344, 344, 344]), array([0, 1, 2]))

In [21]:
prompt_values = {
    'features': X_train[features].iloc[0].to_dict()
    , 'label': y_train[0]
    , 'prediction': rf_classifier.predict(pd.DataFrame([X_train[features].iloc[0].to_dict()]))[0]
    , 'shap_values': zip(features, explainer.shap_values(pd.DataFrame([X_train[features].iloc[0].to_dict()]))[:,:,1])
    , 'embedding': []
}

In [34]:
rf_classifier.predict(pd.DataFrame([X_train[features].iloc[0].to_dict()]))[0]

'benign'

In [ ]:
prompt = f"As a Data Scientist, provide a business analsyis of this dataset about Breast Cancer. Here is some background information. The features for the model when\n {prompt_values['features']} with the SHAP values for {prompt_values['shap_values']}. The prediction for the model is {prompt_values['prediction']}. Provide an analysis of why the model made its decision. Return the answer in markdown."

In [ ]:
role = f'As a Data Scientist, provide an analysis of model {model_name} that uses the dataset {dataset_name}'

In [ ]:
feature_prompt = f'The features for the model are {prompt_values["features"}. Provide a best explaination of the features.'
shap_prompt = f'The shap value for the features are {prompt_values["shap_values"]} where positives pushes to the class {positive_class}.'
labels = f'The model prediction was {prompt_values["prediction"]} with an actual of {prompt_values["label"]}'

In [ ]:
def create_prompt(model_name, dataset_name, positive_class, prompt_values):
    role = f'As a Data Scientist, provide a summary analysis of model {model_name} that uses the dataset {dataset_name}.'
    feature_prompt = f'The features for the model are {prompt_values["features"]}.'
    shap_prompt = f'The shap value for the features are {prompt_values["shap_values"]} where positives pushes to the class {positive_class}. \
    The analysis should include only he most impactful features measured by the SHAP values. Include the SHAP value and the feature value.'
    labels = f'The model prediction was {prompt_values["prediction"]} with an actual of {prompt_values["label"]}.'
    embedding = f'The closest other prediction to the current one has the feature values: {prompt_values["embedding"]}. Provide some context in relation to this prediction'

    return role + ' ' + feature_prompt + ' ' + shap_prompt + ' ' + labels + ' ' + embedding

In [ ]:
prompt = create_prompt('breast-cancer-classification', 'breast cancer', 'malignant', prompt_values)
prompt

In [ ]:
prompt_str = f""

In [ ]:
completion = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
      {
        "role": "user",
        "content": prompt_str
      }
    ],
    temperature=1,
    max_completion_tokens=800,
    top_p=1,
    # reasoning_effort="low",
    stream=True,
    stop=None
)

for chunk in completion:
    print(chunk.choices[0].delta.content or "", end="")

In [52]:
X_train_embeddings.index(target)

AttributeError: 'numpy.ndarray' object has no attribute 'index'

(array([409, 409, 409]), array([0, 1, 2]))

In [54]:
X_train_embeddings[409]

array([-5.5748  ,  8.880056,  5.838913], dtype=float32)